# 모델 2개 baseline 성능 확인

이 노트북은 현재 프로젝트의 2-stage baseline을 확인한다.

- 모델 1: `사고발생 = 위험도 > 0` 분류
- 모델 2: `위험도 > 0` 데이터만 사용한 위험도 회귀
- 학습을 자동으로 실행하지 않는다.
- 기존 metrics/predictions 산출물을 읽어 핵심 지표와 간단한 시각화를 출력한다.

In [ ]:
from pathlib import Path
import os

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'streamlit_app.py').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

In [ ]:
import json
import math
import shlex
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, mean_absolute_error, mean_squared_error

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
REPORTS_DIR = PROJECT_ROOT / 'artifacts' / 'reports'
PREDICTIONS_DIR = PROJECT_ROOT / 'artifacts' / 'predictions'
MODELS_DIR = PROJECT_ROOT / 'artifacts' / 'models'

classifier_metrics_path = REPORTS_DIR / 'mlp_accident_classifier_notebook_metrics.json'
classifier_predictions_path = PREDICTIONS_DIR / 'mlp_accident_classifier_notebook_test_predictions.csv'
classifier_history_path = REPORTS_DIR / 'mlp_accident_classifier_notebook_history.csv'

regressor_metrics_path = REPORTS_DIR / 'mlp_positive_risk_regressor_notebook_metrics.json'
regressor_predictions_path = PREDICTIONS_DIR / 'mlp_positive_risk_regressor_notebook_test_predictions.csv'
regressor_history_path = REPORTS_DIR / 'mlp_positive_risk_regressor_notebook_history.csv'

paths = [
    ('model1 metrics', classifier_metrics_path),
    ('model1 predictions', classifier_predictions_path),
    ('model1 history', classifier_history_path),
    ('model2 metrics', regressor_metrics_path),
    ('model2 predictions', regressor_predictions_path),
    ('model2 history', regressor_history_path),
]
file_status = pd.DataFrame([
    {'구분': name, '경로': path.relative_to(PROJECT_ROOT), '존재': path.exists()} for name, path in paths
])
display(file_status)

## 선택 실행: 모델 1/모델 2 학습

아래 셀은 기본적으로 학습을 실행하지 않는다. 새 산출물이 필요하면 `RUN_TRAINING = True`로 바꾼 뒤 실행한다.

In [ ]:
RUN_TRAINING = False

train_model1_command = [
    'python', 'scripts/train/train_accident_classifier.py',
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
    '--threshold', '0.5',
    '--model-path', str(MODELS_DIR / 'mlp_accident_classifier_notebook.keras'),
    '--metrics-path', str(classifier_metrics_path),
    '--history-path', str(classifier_history_path),
    '--predictions-path', str(classifier_predictions_path),
    '--top-k', '100', '300', '500', '700', '1000',
]
train_model2_command = [
    'python', 'scripts/train/train_positive_risk_regressor.py',
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
    '--model-path', str(MODELS_DIR / 'mlp_positive_risk_regressor_notebook.keras'),
    '--metrics-path', str(regressor_metrics_path),
    '--history-path', str(regressor_history_path),
    '--predictions-path', str(regressor_predictions_path),
]

print('모델 1 실행 명령:')
print(shlex.join(train_model1_command))
print('\n모델 2 실행 명령:')
print(shlex.join(train_model2_command))

if RUN_TRAINING:
    subprocess.run(train_model1_command, cwd=PROJECT_ROOT, check=True)
    subprocess.run(train_model2_command, cwd=PROJECT_ROOT, check=True)
else:
    print('RUN_TRAINING=False 이므로 학습을 실행하지 않았습니다.')

In [ ]:
required_paths = [classifier_metrics_path, classifier_predictions_path, regressor_metrics_path, regressor_predictions_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_text = '\n'.join(str(path.relative_to(PROJECT_ROOT)) for path in missing_paths)
    raise FileNotFoundError(f'필수 산출물이 없습니다. 먼저 학습을 실행하십시오.\n{missing_text}')

with classifier_metrics_path.open('r', encoding='utf-8') as file:
    classifier_metrics = json.load(file)
with regressor_metrics_path.open('r', encoding='utf-8') as file:
    regressor_metrics = json.load(file)

classifier_predictions = pd.read_csv(classifier_predictions_path)
regressor_predictions = pd.read_csv(regressor_predictions_path)

print(f'model1 rows: {len(classifier_predictions):,}')
print(f'model2 rows: {len(regressor_predictions):,}')
display(classifier_predictions.head())
display(regressor_predictions.head())

In [ ]:
def precision_recall_at_top_percent(y_true, score, top_percent=0.10):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score, dtype=float)
    k = max(1, math.ceil(len(score) * top_percent))
    order = np.argsort(-score, kind='mergesort')[:k]
    hits = int(y_true[order].sum())
    positives = int(y_true.sum())
    return {
        'top_percent': top_percent,
        'k': k,
        'hit_count': hits,
        'precision': hits / k,
        'recall': hits / positives if positives else np.nan,
    }

y_true_model1 = classifier_predictions['사고발생'].to_numpy(dtype=int)
score_model1 = classifier_predictions['사고발생확률'].to_numpy(dtype=float)
pred_model1 = classifier_predictions['사고발생예측'].to_numpy(dtype=int)
model1_top10 = precision_recall_at_top_percent(y_true_model1, score_model1, top_percent=0.10)

model1_summary = pd.DataFrame([{
    'experiment': 'two_stage_baseline_model1',
    'pr_auc': average_precision_score(y_true_model1, score_model1),
    'f1': f1_score(y_true_model1, pred_model1, zero_division=0),
    'precision@top10%': model1_top10['precision'],
    'recall@top10%': model1_top10['recall'],
    'top10_k': model1_top10['k'],
}])

actual_risk_model2 = regressor_predictions['위험도'].to_numpy(dtype=float)
pred_risk_model2 = regressor_predictions['pred_위험도'].to_numpy(dtype=float)
model2_summary = pd.DataFrame([{
    'experiment': 'two_stage_baseline_model2',
    'mae': regressor_metrics.get('risk_scale', {}).get('mae', mean_absolute_error(actual_risk_model2, pred_risk_model2)),
    'rmse': regressor_metrics.get('risk_scale', {}).get('rmse', math.sqrt(mean_squared_error(actual_risk_model2, pred_risk_model2))),
    'log_mae': regressor_metrics.get('log_scale', {}).get('mae', np.nan),
    'log_rmse': regressor_metrics.get('log_scale', {}).get('rmse', np.nan),
}])

display(model1_summary)
display(model2_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model1_plot = model1_summary.set_index('experiment')[['pr_auc', 'f1', 'precision@top10%', 'recall@top10%']].T
model1_plot.plot(kind='bar', ax=axes[0], legend=False, color='#2563eb')
axes[0].set_ylim(0, 1)
axes[0].set_title('Model 1 Classification Metrics')
axes[0].tick_params(axis='x', rotation=45)

sample = regressor_predictions.sample(min(len(regressor_predictions), 5000), random_state=42)
axes[1].scatter(sample['위험도'], sample['pred_위험도'], s=10, alpha=0.35, color='#dc2626')
axes[1].set_title('Model 2 Actual vs Predicted Risk')
axes[1].set_xlabel('actual 위험도')
axes[1].set_ylabel('pred 위험도')

plt.tight_layout()
plt.show()

In [ ]:
history_paths = [
    ('model1', classifier_history_path),
    ('model2', regressor_history_path),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, (name, path) in zip(axes, history_paths):
    if not path.exists():
        axis.set_title(f'{name}: history 없음')
        axis.axis('off')
        continue
    history = pd.read_csv(path)
    loss_columns = [column for column in ['loss', 'val_loss'] if column in history.columns]
    if loss_columns:
        history[loss_columns].plot(ax=axis)
        axis.set_title(f'{name} Training History')
        axis.set_xlabel('epoch')
        axis.set_ylabel('loss')
    else:
        axis.set_title(f'{name}: loss 컬럼 없음')
        axis.axis('off')

plt.tight_layout()
plt.show()